# Week 4: What is an agent harness?

*Course project: a personal assistant for **Alex, 44**: senior backend developer, full-time job, two kids,
retraining as an **AI engineer** while job-hunting.*

This notebook follows the talk **"What Is an Agent Harness?"** slide by slide. For three weeks we built parts of a
harness without calling it that: the loop (week 1), tools and permissions (week 2), the desk (week 3). Today we put
them together into **one engine** and open **three doors** into it.

> **The model is the engine. The harness is the car.** The model reads text and writes text. The harness does
> everything else: the loop, the tools, the memory, the safety rules, the screens. *Same model, better harness,
> better agent.*

| Talk | Part of the harness | Section |
|---|---|---|
| 01 The big idea (slides 2–3) | same model, with and without a harness | 1 |
| 02 The map (slides 4, 10) | a **headless** engine: messages in, events out | 2 |
| 03 Engine room (slides 5–6) | the agent loop, turn by turn, with limits | 3 |
| 03 Engine room (slides 7–8) | the desk and compaction: `[summary]` + `[tail]` | 4 |
| 03 Engine room (slide 9) | modules: permissions, sandbox, skills, memory, providers | 5 |
| 04 The doors (slides 11–12) | interactive (steering queue, priority gate) · remote (CLI, cron, webhook, in parallel) | 6 |
| 04 The doors (slide 13) | evals and observability: flight recorder, replay, regressions | 7 |
| 05 Any agent (slide 14) | the same harness runs a support agent | 8 |
| 06 Case study (slide 16) | the coding-agent diagram, mapped onto our code | 9 |

⏱ **Core: sections 1–3, 5, 6, 7, 8 (~75 min).** Cells marked *optional* are for after class.

## 0. Setup

You did the setup at home (`SETUP.md`); this checks it again. This week runs on **OpenAI** (`gpt-6-luna`).

In [ ]:
%run ../check_setup.py --no-api

In [ ]:
import itertools, json, queue, re, threading, time
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field
from datetime import timedelta
from pathlib import Path
from typing import Any, Callable, Iterator

from IPython.display import Markdown, display
from pydantic import Field

import lifedb, life_tools
from agent_core import DESTRUCTIVE, READ, WRITE, execute_tool, run_agent, tool, tool_schemas
from life_tools import Args
from providers import ToolCall, ToolResult, load_api_key, open_conversation

PROVIDER, MODEL = "openai", "gpt-6-luna"
load_api_key(PROVIDER)
from openai import OpenAI
client = OpenAI()


def fresh_week() -> None:
    """Alex's week, dated to this week, from scratch (undo everything the agents changed)."""
    lifedb.create_db()
    life_tools.reset()


fresh_week()
WEEK = lifedb.week()
ALL_TOOLS = {**life_tools.TOOLS, **life_tools.WRITE_TOOLS}          # read + write: calendar, tasks, email

SYSTEM = (
    "You are Alex's personal assistant. Alex is 44, a senior backend developer with a full-time job and two kids, "
    f"retraining as an AI engineer and job-hunting. Today is {WEEK.today}; this week runs to {WEEK.label(6)}. "
    "All times are Eastern. Use the tools to check facts; never guess. Keep the final answer short."
)
print("Today is", WEEK.today, "| tools:", ", ".join(ALL_TOOLS))

## 1. The model is the engine, the harness is the car (slides 2–3)

Same model, same question, twice:

1. **The engine alone**: one call, no tools. The model can't open Alex's calendar.
2. **The car we built in weeks 1–3**: `run_agent`, the loop with Alex's tools.

Brightline Health offered three interview times. Only one fits Alex's rules (work 9:00–17:30, no calls 17:30–20:00).

In [ ]:
QUESTION = (f"Brightline Health offered a 45-minute interview: {WEEK.label(2)} 12:00, {WEEK.label(3)} 18:00 or "
            f"{WEEK.label(4)} 08:00. Which one fits my calendar and my rules? One sentence.")

engine = open_conversation(PROVIDER, MODEL, QUESTION, system="You are Alex's personal assistant.", effort="low")
print("🔩 The engine alone:\n  ", engine.ask().text)

car = run_agent(QUESTION, life_tools.TOOLS, system=SYSTEM, effort="low", verbose=False)
print(f"\n🚗 The engine in a harness ({car.turns} turns, tools: {', '.join(car.tool_calls)}):\n  ", car.answer)

The engine alone has to guess, or asks you to look it up. The harness checked the calendar and the preferences note,
so it answers *Friday 08:00*. The model is the same; the harness is the difference.

**The six jobs of a harness (slide 3), and where we already built them:**

| Job | What it does | Where we built it |
|---|---|---|
| Agent loop | keeps the work going: ask the model, do it, report back, repeat | week 1 |
| Context window | decides what the model sees | week 3 |
| Compaction | tidies up when the desk is full | week 3 |
| Modules | tools, memory, skills, sandbox, permissions | weeks 2–3 (skills: today) |
| Interfaces | lets people and systems in | **today** |
| Evals | proves it works | **today** (a first look) and week 5 |

## 2. The map: a headless engine (slides 4, 10)

The harness has three parts: **doors in**, the **engine room**, and the **test track**. The key design choice:
the engine room is **headless**. It has no screen. It takes a task in and sends **events** out:

> `start` → `model` → `action` → `observation` → … → `done`

A notebook prints the events; a chat app streams them to a person; a cron job saves them; an eval scores them.
**Build the engine once, then reuse it behind any door.**

An `Event` is a small record: what happened, the details, which harness sent it (many can run at once), and when.

In [ ]:
@dataclass
class Event:
    kind: str                       # start | model | think | action | approval | observation | steer | compact | limit | done
    data: dict = field(default_factory=dict)
    agent: str = "alex"             # which harness sent it: many can run at the same time
    t: float = field(default_factory=time.time)


@dataclass
class Limits:
    """So a confused model can't loop forever (slide 5): turns, money and time, and the length of one reply."""
    max_turns: int = 12
    max_cost: float = 0.05          # dollars
    max_seconds: float = 240
    max_output_tokens: int = 4_000  # per model call: a runaway reply is cut off instead of running for minutes

    def broken(self, turn: int, cost: float, seconds: float) -> str | None:
        if turn >= self.max_turns:
            return f"{self.max_turns} turns"
        if cost > self.max_cost:
            return f"cost ${cost:.4f} > ${self.max_cost}"
        if seconds > self.max_seconds:
            return f"{seconds:.0f} s > {self.max_seconds} s"
        return None


PRICE = {"input": 0.10, "cached": 0.01, "output": 0.50}      # gpt-6-luna, $ per 1M tokens


def step_cost(step) -> float:
    uncached = step.input_tokens - step.cached_tokens
    return (uncached * PRICE["input"] + step.cached_tokens * PRICE["cached"] + step.output_tokens * PRICE["output"]) / 1e6

The harness needs to know three things about every tool call. Its **policy** answers *may this run?*
(allow / ask / deny, from week 2). Its **approver** answers *what does the human say?* Its **sandbox** answers
*what would change?* Here are the simplest versions; section 5 plugs in better ones.

In [ ]:
def policy_by_effect(rules: dict | None = None, overrides: dict | None = None) -> Callable[[str, str], str]:
    """Week 2's allow / ask / deny: by the tool's effect, with per-tool overrides."""
    rules = rules or {READ: "allow", WRITE: "ask", DESTRUCTIVE: "ask"}
    overrides = overrides or {}
    return lambda name, effect: overrides.get(name, rules.get(effect, "ask"))     # unknown effect: be careful


def always(answer: str) -> Callable[[Event], str]:
    """An approver that always gives the same answer: 'yes', 'no', or 'hold' (decide later)."""
    return lambda event: answer


def ask_human(event: Event) -> str:
    print(f"\n🔐 {event.data['tool']} wants to make this change:\n{event.data['preview']}")
    return "yes" if input("Allow? [y/N] ").strip().lower() in ("y", "yes") else "no"


@dataclass
class Box:
    """What a sandbox hands back: a preview for the human, maybe the result already, and keep or not."""
    preview: str
    result: ToolResult | None = None
    keep: bool = False


def fmt_args(args: dict) -> str:
    return ", ".join(f"{k}={v!r}" for k, v in args.items())


@contextmanager
def no_sandbox(call, tools):
    """Nothing to dry-run: the human sees the call itself, and it runs only after a yes."""
    yield Box(preview=f"  {call.name}({fmt_args(call.input)})")

And here is the engine room: **the whole harness in one class**. Read `run()` from top to bottom: it is week 1's loop,
plus places where the other parts plug in. Everything it does becomes an event: `yield` hands the event to whoever
is listening and waits until they ask for the next one.

- **modules** add tools and instructions (section 5)
- **policy, approver, sandbox** decide every action that changes something (sections 3, 5)
- **compactor** tidies the desk before each model call (section 4)
- **steering** lets the user's new messages in between steps (section 6)
- **limits** stop runaway runs (section 3)

In [ ]:
@dataclass
class Held:
    """A change that waits for a human (the remote door: nobody is at the keyboard)."""
    agent: str
    tool: str
    args: dict
    preview: str


class Harness:
    """The engine room: loop + context + modules. It has no screen of its own: `run()` yields Events,
    and whoever started it (a notebook, a CLI, a cron job, an eval) decides what to do with them."""

    def __init__(self, tools: dict, *, system: str, name: str = "alex", model: str = MODEL,
                 provider: str = PROVIDER, effort: str | None = "low", limits: Limits | None = None,
                 modules: list[dict] = (), policy=None, approver=None, sandbox=None,
                 compactor=None, steering=None, extra: dict | None = None, client=None):
        self.tools, self.system = dict(tools), system
        for m in modules:                                   # plug-in parts: more tools and/or instructions
            self.tools |= m.get("tools", {})
            if m.get("system"):
                self.system += "\n\n" + m["system"]
        self.name, self.model, self.provider, self.effort = name, model, provider, effort
        self.limits = limits or Limits()
        self.policy = policy or policy_by_effect()
        self.approver = approver or ask_human
        self.sandbox = sandbox or no_sandbox
        self.compactor, self.steering = compactor, steering
        self.extra, self.client = extra, client
        self.held: list[Held] = []                          # changes waiting for a human
        self.conv = None                                    # the desk of the last run, to look at afterwards

    def event(self, kind: str, **data) -> Event:
        return Event(kind, data, self.name)

    def run(self, task: str) -> Iterator[Event]:
        conv = self.conv = open_conversation(self.provider, self.model, task, tools=tool_schemas(self.tools),
                                             system=self.system, effort=self.effort, client=self.client,
                                             extra=self.extra)
        conv.max_tokens = self.limits.max_output_tokens
        cost, t0 = 0.0, time.time()
        yield self.event("start", task=task, model=self.model, tools=list(self.tools))

        for turn in range(1, self.limits.max_turns + 1):
            if self.steering:                               # the safe moment: between two steps
                for msg in self.steering.let_in():
                    yield self.event("steer", turn=turn, text=msg.text, urgent=msg.urgent)
                    if msg.is_stop:
                        yield self.event("done", answer="[stopped by the user]", stop="user", turns=turn - 1,
                                         cost=cost, seconds=time.time() - t0)
                        return
                    conv.add_user_message(msg.text)
            if self.compactor and (info := self.compactor(conv)):
                yield self.event("compact", turn=turn, **info)

            step = conv.ask()                               # THINK: the model picks the next step
            cost += step_cost(step)
            yield self.event("model", turn=turn, input=step.input_tokens, cached=step.cached_tokens,
                             output=step.output_tokens, cost=cost)

            if step.stop != "tool_use":                     # the model says "done" (or can't go on)
                if self.steering and self.steering.waiting():
                    continue                                # the user typed while it answered: keep going
                yield self.event("done", answer=step.text, stop=step.stop, turns=turn, cost=cost,
                                 seconds=time.time() - t0)
                return
            if step.text:
                yield self.event("think", turn=turn, text=step.text)

            results = []
            for call in step.tool_calls:                    # ACT: the harness runs the tools, not the model
                yield self.event("action", turn=turn, tool=call.name, args=call.input)
                result = yield from self.act(call, turn)
                yield self.event("observation", turn=turn, tool=call.name, text=result.content,
                                 error=result.is_error)     # LOOK: the result goes back as an observation
                results.append(result)
            conv.add_tool_results(results)

            if reason := self.limits.broken(turn, cost, time.time() - t0):
                break
        else:
            reason = f"{self.limits.max_turns} turns"
        yield self.event("limit", turn=turn, reason=reason)
        yield self.event("done", answer=f"[stopped: {reason}]", stop="limit", turns=turn, cost=cost,
                         seconds=time.time() - t0)

    def act(self, call, turn: int):
        """One tool call, through the permission rules. Yields an 'approval' event when a human must decide."""
        entry = self.tools.get(call.name)
        rule = self.policy(call.name, entry["effect"]) if entry else "allow"   # unknown tool: execute_tool explains
        if rule == "allow":
            return execute_tool(call, self.tools)
        if rule == "deny":
            return ToolResult(call.id, f"'{call.name}' is turned off for this assistant. "
                                       "Tell the user what you wanted to do, so they can do it themselves.", True)

        with self.sandbox(call, self.tools) as box:          # "ask": try it safely, show the change, wait
            if box.result is not None and box.result.is_error:
                return box.result                           # an invalid call never bothers the human
            ask = self.event("approval", turn=turn, tool=call.name, args=call.input, preview=box.preview)
            yield ask
            answer = self.approver(ask)
            box.keep = answer == "yes"
        if answer == "yes":
            return box.result or execute_tool(call, self.tools)
        if answer == "hold":
            self.held.append(Held(self.name, call.name, call.input, box.preview))
            return ToolResult(call.id, "Held for approval: Alex reviews held changes in the morning. Treat it as "
                                       "pending, don't retry it, and say in your answer what is waiting for him.")
        return ToolResult(call.id, "The user declined this change. Ask them what they would prefer.", True)

The engine has no screen, so we need something that *shows* events. `show` prints one event; `run` drives a harness,
shows every event as it happens and returns them all; `turn_table` draws the run like slide 6.

In [ ]:
ICON = {"start": "▶", "think": "💭", "action": "🔧", "approval": "🔐", "observation": "👀", "steer": "📨",
        "compact": "🗜", "limit": "⛔", "done": "🏁"}
PRINT = threading.Lock()                         # several harnesses may print at once


def short(x: Any, n: int = 140) -> str:
    s = str(x).replace("\n", " | ")
    return s if len(s) <= n else s[:n] + "…"


def show(ev: Event, *, tag: bool = False) -> None:
    d, who = ev.data, f"[{ev.agent}] " if tag else ""
    line = {
        "start": lambda: f"task: {short(d['task'], 100)}",
        "model": lambda: f"turn {d['turn']} · desk {d['input']:,} tokens · total ${d['cost']:.4f}",
        "think": lambda: short(d["text"]),
        "action": lambda: f"{d['tool']}({short(fmt_args(d['args']), 110)})",
        "approval": lambda: f"paused: a human must approve {d['tool']}\n{d['preview']}",
        "observation": lambda: ("❌ " if d["error"] else "") + short(d["text"]),
        "steer": lambda: ("URGENT: " if d["urgent"] else "") + d["text"],
        "compact": lambda: f"desk {d['before']:,} → {d['after']:,} tokens ({d['summarised']} items summarised, "
                           f"{d['kept']} kept word for word)",
        "limit": lambda: f"limit reached: {d['reason']}",
        "done": lambda: f"{d['turns']} turns · ${d['cost']:.4f} · {d['seconds']:.0f} s · stop={d['stop']}\n"
                        f"   {d['answer']}",
    }[ev.kind]()
    indent = "" if ev.kind in ("start", "model", "done") else "   "
    with PRINT:
        print(f"{who}{indent}{ICON.get(ev.kind, '·')} {line}")


def run(harness: Harness, task: str, *, quiet: bool = False, tag: bool = False, on_event=None) -> list[Event]:
    events = []
    for ev in harness.run(task):
        events.append(ev)
        if not quiet:
            show(ev, tag=tag)
        if on_event:
            on_event(ev)
    return events


def final(events: list[Event]) -> str:
    return next((e.data["answer"] for e in reversed(events) if e.kind == "done"), "")


def turn_table(events: list[Event]) -> None:
    """Slide 6: one row per action, what the model decided and what the tool reported back."""
    rows, paused = [], ""
    for e in events:
        d = e.data
        if e.kind == "action":
            rows.append([str(d["turn"]), f"`{d['tool']}`({short(fmt_args(d['args']), 70)})", ""])
            paused = ""
        elif e.kind == "approval":
            paused = "⏸ paused for a human → "
        elif e.kind == "observation" and rows:
            rows[-1][2] = paused + short(d["text"], 110).replace("|", "/")
        elif e.kind == "steer":
            rows.append([str(d["turn"]), "📨 *the user:* " + short(d["text"], 80), "let in between steps"])
        elif e.kind == "compact":
            rows.append([str(d["turn"]), "🗜 *the harness compacts*", f"{d['before']:,} → {d['after']:,} tokens"])
        elif e.kind == "done":
            rows.append([str(d["turns"]), "answer, then stop", short(d["answer"], 110).replace("|", "/")])
    table = ["| Turn | The model decides (action) | The tool reports back (observation) |", "|---|---|---|"]
    display(Markdown("\n".join(table + ["| " + " | ".join(r) + " |" for r in rows])))

## 3. The agent loop: think, act, look, repeat (slides 5–6)

**The model never executes anything.** It only *asks* for an action. The harness runs it and returns the
observation. That is why the harness is where the approval step and the log live.

Slide 6 showed a support agent's run, turn by turn. Here is Alex's version. Overnight, Northwind's recruiter wrote:
the recruiter screen on Tuesday 20:30 overlaps Alex's evening class. Alex's harness: all his tools, writes need
approval (`ask`), and for now the approver says yes to everything.

In [ ]:
NORTHWIND = next(e for e in life_tools.inbox() if e["id"] == "northwind")
TASK = (f"Email from {NORTHWIND['from']}:\n\n{NORTHWIND['body']}\n\n"
        "Move the screen to a time that fits my rules, and draft a reply to Dana.")

fresh_week()
alex = Harness(ALL_TOOLS, system=SYSTEM, approver=always("yes"),
               policy=policy_by_effect(overrides={"draft_email": "allow"}))   # a draft sends nothing: allow it
events = run(alex, TASK)

In [ ]:
turn_table(events)

Read the table like slide 6:

- Every turn is one trip around the loop: the model decides, the harness acts, the observation goes back.
  (A turn can have several actions: the model may ask for a few tools at once.)
- **⏸ paused for a human**: that's the harness, not the model. A permission rule (`write → ask`) stopped the step.
- The last row: **deciding to stop is also a decision.**

Our `draft_email` ran without asking: a draft sends nothing. `send_email` would have asked. Which actions of
*your* agent should never run without a human saying yes? (Slide 17, question 3.)

### Limits

A while loop with a model in it needs brakes. Same harness, with a budget too small for the job:

In [ ]:
fresh_week()
tight = Harness(ALL_TOOLS, system=SYSTEM, approver=always("yes"), limits=Limits(max_turns=2))
events = run(tight, TASK)

The run stops cleanly at the limit, with an event saying why, instead of looping (or spending) forever. In production
you set all of them: turns, cost, time. `Limits(max_cost=0.001)` and `Limits(max_seconds=5)` work the same way.

The fourth limit, `max_output_tokens`, works **inside** one model call. While testing this notebook, a model call
twice "ran away": it wrote 16,000 tokens of reasoning, took up to five minutes, and produced no answer. The other limits
only check *between* steps, so they couldn't help. A cap on each reply cuts a runaway off early (`stop=max_tokens`).

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 4 (the same code as the cells above), so you can continue here.
# from checkpoint import *

## 4. The desk, and compaction when it's full (slides 7–8)

**The model only knows what's on the desk.** The harness decides what goes on it. Week 3 was all about this; here we
plug it into the harness as one part: a **compactor** that runs before every model call and asks *should compact?*

When the desk passes a threshold, the old part becomes a **`[summary]`**, the last few model steps stay **word for
word** (the **`[tail]`**), and the task stays pinned on top. *Like meeting minutes: keep the decisions, drop the chatter.*

One detail matters: the tail must start where a model step starts, so no tool result is ever separated from the
call that asked for it.

In [ ]:
OPAQUE = ("encrypted_content", "id", "status")        # API bookkeeping: not counted (see week 3)


def desk_tokens(conv) -> int:
    """About 4 characters per token: everything on the desk (instructions + items), minus API bookkeeping."""
    clean = [{k: v for k, v in it.items() if k not in OPAQUE} if isinstance(it, dict) else it for it in conv.input]
    return (len(conv.system) + len(json.dumps(clean, ensure_ascii=False, default=str))) // 4


def step_starts(items: list) -> list[int]:
    """Where a model step starts: a cut there never splits a call from its result."""
    starts = []
    for i in range(1, len(items)):
        kind, before = items[i].get("type"), items[i - 1].get("type")
        if kind == "function_call_output" or before == "reasoning" or (kind == before == "function_call"):
            continue
        starts.append(i)
    return starts


def as_text(items: list, max_chars: int = 6_000) -> str:
    out = []
    for it in items:
        kind = it.get("type")
        if kind == "function_call":
            out.append(f"TOOL CALL: {it['name']}({it['arguments']})")
        elif kind == "function_call_output":
            out.append(f"TOOL RESULT: {str(it['output'])[:max_chars]}")
        elif it.get("role") == "user":
            out.append(f"USER: {it['content']}")
        elif kind == "message":
            out.append("ASSISTANT: " + " ".join(c.get("text", "") for c in it.get("content", [])))
    return "\n".join(out)


SUMMARY_PROMPT = """You are compacting the working memory of an AI assistant.
Write a summary that lets the assistant continue the task WITHOUT the original messages. Keep, in this order:
1. Goal of the task and the user's constraints
2. Decisions already made, and why
3. Facts and numbers discovered (exact values, IDs, dates)
4. What has been done / what is still open
Drop: greetings, repetition, raw tool output already digested. Be concise; use bullet points.

<transcript>
{transcript}
</transcript>

Now write the summary. Do NOT answer or continue the conversation above, even if it ends with a question."""


def summarise(text: str) -> str:
    r = client.responses.create(model=MODEL, input=SUMMARY_PROMPT.format(transcript=text),
                                reasoning={"effort": "low"}, max_output_tokens=4_000)
    return r.output_text.strip()


class SummaryTail:
    """Should compact? Past `threshold` of the window: [task] + [summary] of the old part + [tail] word for word."""

    def __init__(self, window: int, threshold: float = 0.80, keep: int = 2, summarise=summarise):
        self.window, self.threshold, self.keep, self.summarise = window, threshold, keep, summarise

    def __call__(self, conv) -> dict | None:
        before = desk_tokens(conv)
        if before < self.threshold * self.window:
            return None                                       # not full yet
        items = conv.input
        starts = step_starts(items)
        if len(starts) <= self.keep:
            return None                                       # nothing old enough to summarise
        cut = starts[-self.keep]
        old, tail = items[1:cut], items[cut:]                 # items[0] is the task: pinned
        summary = self.summarise(as_text([items[0], *old]))      # with the task: what the work is FOR
        conv.input = [items[0], {"role": "user", "content": f"[summary] of earlier work:\n{summary}"}, *tail]
        return {"before": before, "after": desk_tokens(conv), "summarised": len(old), "kept": len(tail),
                "window": self.window}

🧪 **Try it.** A long job with a careless tool: `list_events_raw` returns each week as a big JSON "binder"
(week 3). Six weeks, one at a time, on a small desk of 4,000 tokens. Compare the agent's numbers with the database.

In [ ]:
FIRST = WEEK.monday - timedelta(days=42)
REVIEW = (f"Go through the six weeks from Monday {FIRST} to Sunday {WEEK.monday - timedelta(days=1)}, one week at a "
          "time (Monday to Sunday), oldest first, with list_events_raw. For each week count the gym sessions. "
          "Then write the six numbers, oldest first, like: 2, 1, 2, 0, 2, 1.")


def gym_truth(weeks: int = 6) -> list[int]:
    out = []
    for k in range(weeks, 0, -1):
        monday = WEEK.monday - timedelta(days=7 * k)
        rows = life_tools.db_rows("SELECT COUNT(*) AS n FROM events WHERE title = 'Gym' AND date BETWEEN ? AND ?",
                                  (monday.isoformat(), (monday + timedelta(days=6)).isoformat()))
        out.append(rows[0]["n"])
    return out


fresh_week()
WINDOW = 4_000
desk = Harness(life_tools.BINDER_TOOLS, system=SYSTEM, compactor=SummaryTail(window=WINDOW),
               extra={"parallel_tool_calls": False}, limits=Limits(max_turns=14))
events = run(desk, REVIEW)
print("\nTruth from the database:", ", ".join(map(str, gym_truth())))

In [ ]:
def desk_bars(events, window):
    """Slide 7's 'desk space used', per model call. The API's own count of the input tokens."""
    for e in events:
        if e.kind == "compact":
            print(f"      🗜 compact: {e.data['before']:,} → {e.data['after']:,} tokens")
        if e.kind == "model":
            used = e.data["input"] / window
            print(f"turn {e.data['turn']:>2} {'█' * min(40, round(used * 20)):<40} {used:>5.0%}")


desk_bars(events, WINDOW)
print("\nThe desk now starts with:\n", short(desk.conv.input[1].get("content", ""), 400))

What to look for:

- The desk grows by one binder per turn, until **should compact? → yes**. Then it drops (slide 8: *95% → 35%*).
- The **`[summary]`** carries the counts already found; the **`[tail]`** keeps the last steps word for word.
- Are the six numbers still right? *The risk is a summary that forgets something important.* That's why long agent
  sessions sometimes "forget" an instruction from an hour ago: it was lost in the summary.

Our estimate (characters ÷ 4) and the API's count (`model` events) differ a little. That's fine for a threshold.

**A bug we hit while writing this notebook.** The first version of `SummaryTail` didn't pass the task to the summariser
(the task is pinned, so it seemed unnecessary). The summary then started with *"The original request is missing, so
the intended task is unknown"*, and after every compaction the agent started over: it fetched the same weeks again and
again until the turn limit stopped it. **The summariser must know what the work is for.** Remove `items[0], ` from
`as_text([items[0], *old])` and rerun to see it.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 5 (the same code as the cells above), so you can continue here.
# from checkpoint import *

## 5. Modules: plug-in parts around the loop (slide 9)

| Helping the model think | Helping the tools act safely |
|---|---|
| **LLM providers**: swap the model | **Sandbox**: try the change where mistakes stay contained |
| **Memory**: notes that survive between sessions | **Permissions**: what needs a human's OK first |
| **Skills**: instructions for known jobs, loaded only when needed | **Tools & connectors**: the agent's hands |

*Modules* means you can swap each part without rewriting the loop. Our `Harness` takes them as arguments.

### 5a. Sandbox + permissions

In section 3 the human approved a change they couldn't see. Week 2's **dry run** fixes that: run the tool inside a
database transaction, show the difference, then **COMMIT** on a yes or **ROLLBACK** otherwise. Mistakes stay contained.

In [ ]:
@contextmanager
def sqlite_sandbox(call, tools):
    """Run the call inside a transaction, show the diff, COMMIT on a yes, ROLLBACK otherwise."""
    with life_tools.LOCK:                              # one change at a time, even with many harnesses
        conn = life_tools.rw()
        before = lifedb.snapshot(conn)
        conn.execute("BEGIN")
        box = Box(preview="")
        try:
            box.result = execute_tool(call, tools)
            box.preview = lifedb.diff(before, lifedb.snapshot(conn))
            yield box
        finally:
            conn.execute("COMMIT" if box.keep else "ROLLBACK")


ALEX_POLICY = policy_by_effect(overrides={"draft_email": "allow", "remember": "allow"})

In [ ]:
def reviewer(event: Event) -> str:
    """A scripted Alex: approves calendar changes, never lets the assistant send email for him."""
    verdict = "no" if event.data["tool"] == "send_email" else "yes"
    print(f"   🙋 Alex looks at the change above and says: {verdict}")
    return verdict


fresh_week()
alex = Harness(ALL_TOOLS, system=SYSTEM, policy=ALEX_POLICY, approver=reviewer, sandbox=sqlite_sandbox)
events = run(alex, TASK + " Then send it.")
print("\nOutbox:", [(m["status"], m["subject"]) for m in life_tools.outbox()])

Now Alex sees **exactly** what will change (`- old row` / `+ new row`) before saying yes. When he says no to
`send_email`, nothing reaches the database (ROLLBACK) and the model is told so, so it can adapt: the draft stays a draft.

*Try it yourself:* replace `approver=reviewer` with `approver=ask_human` and answer the prompts.

### 5b. Skills: ready-made instructions, loaded only when needed

Alex has a few jobs that come back again and again: replying to recruiters, rescheduling, the Sunday review. Each one
has a page of instructions in `skills/<name>/SKILL.md`. Putting every page in the system prompt costs tokens on every
call, for jobs that aren't happening. Instead, the system prompt gets a **menu** (one line per skill), and a
`load_skill` tool puts the full page on the desk **when the job starts**.

In [ ]:
print(Path('skills/reply-to-recruiter/SKILL.md').read_text())

In [ ]:
def read_skill(path: Path) -> dict:
    """SKILL.md: a small header (name, description) between --- lines, then the instructions."""
    _, header, body = path.read_text(encoding="utf-8").split("---", 2)
    meta = dict(line.split(":", 1) for line in header.strip().splitlines())
    return {"name": meta["name"].strip(), "description": meta["description"].strip(), "body": body.strip()}


def skills_module(folder: str = "skills") -> dict:
    skills = {s["name"]: s for s in map(read_skill, sorted(Path(folder).glob("*/SKILL.md")))}

    class LoadSkillIn(Args):
        name: str = Field(json_schema_extra={"enum": list(skills)})

    def load_skill(name: str) -> str:
        if name not in skills:
            raise ValueError(f"No skill '{name}'. Skills: {', '.join(skills)}")
        return skills[name]["body"]

    menu = "\n".join(f"- {s['name']}: {s['description']}" for s in skills.values())
    return {
        "system": "Skills: step-by-step instructions for jobs Alex asks for often. When a task matches one, "
                  "call load_skill with its name FIRST, then follow it.\n" + menu,
        "tools": {"load_skill": tool(load_skill, input_model=LoadSkillIn, description=
                                     "Load the full instructions of one skill from the menu in your instructions.")},
        "skills": skills,
    }


SKILLS = skills_module()

In [ ]:
every_page = sum(len(s["body"]) for s in SKILLS["skills"].values()) // 4
print(f"Menu in the system prompt: ~{len(SKILLS['system']) // 4} tokens on every call "
      f"(all {len(SKILLS['skills'])} pages would be ~{every_page} tokens on every call)\n")
print(SKILLS["system"])

In [ ]:
BRIGHTLINE = next(e for e in life_tools.inbox() if e["id"] == "brightline")
fresh_week()
alex = Harness(ALL_TOOLS, system=SYSTEM, modules=[SKILLS], policy=ALEX_POLICY, approver=always("yes"),
               sandbox=sqlite_sandbox)
events = run(alex, f"Email from {BRIGHTLINE['from']}:\n\n{BRIGHTLINE['body']}\n\nHandle it.")
draft = next((m for m in life_tools.outbox() if m["status"] == "draft"), None)
print("\nThe draft:\n", draft["body"] if draft else "(none)")

The model saw a one-line menu, recognised the job, loaded the skill and followed it: it read the preferences,
booked the slot that fits, saved a **draft** (not an email), signed it the way the skill says. The skill is just a
file: Alex (or you) can improve it without touching the code.

### 5c. Memory: notes that survive between sessions

Week 3 built a full memory drawer. As a module it's small: the notes are put in the instructions when a session
starts, and a `remember` tool adds to them.

In [ ]:
def memory_module(path: str = "memories/alex.md") -> dict:
    file = Path(path)

    class RememberIn(Args):
        fact: str = Field(min_length=3, max_length=300, description="One short fact, in Alex's words")

    def remember(fact: str) -> str:
        file.parent.mkdir(parents=True, exist_ok=True)
        with file.open("a", encoding="utf-8") as f:
            f.write(f"- {fact}\n")
        return "Remembered."

    notes = file.read_text(encoding="utf-8") if file.exists() else "(nothing yet)"
    return {"system": f"What you remember about Alex from earlier sessions (his rules: follow them):\n{notes}\n"
                      "When Alex tells you something to keep for next time, call remember.",
            "tools": {"remember": tool(remember, input_model=RememberIn, effect=WRITE, description=
                                       "Save one fact about Alex for future sessions.")}}

In [ ]:
Path("memories/alex.md").unlink(missing_ok=True)
fresh_week()

# Session 1: Alex tells the assistant something.
monday = Harness(ALL_TOOLS, system=SYSTEM, modules=[memory_module()], policy=ALEX_POLICY, approver=always("yes"))
run(monday, "New rule: no calls on Thursdays, at any time. Thursday is my capstone day. Remember that.", quiet=True)
print("memories/alex.md:\n" + Path("memories/alex.md").read_text())

# Session 2: a new harness (a new day). Only the memory file connects the two.
ask = f"Can I take a 30-minute call with Northwind on {WEEK.label(3)} at 08:00? Yes or no, and why, in one line."
for label, modules in (("without memory", []), ("with memory", [memory_module()])):
    h = Harness(ALL_TOOLS, system=SYSTEM, modules=modules, policy=ALEX_POLICY, approver=always("no"))
    print(f"{label:>15}: {final(run(h, ask, quiet=True))}")

Same model, same question. The calendar says 08:00 is free, so without memory the answer is *yes*. Only the harness
changed: with the memory module, the assistant knows Thursday is the capstone day.

### 5d. LLM providers

The last module is the engine itself: `Harness(..., provider="anthropic", model="claude-sonnet-5-5")` runs the same
harness on Claude (if you have a key), `provider="ollama"` on a local model. Nothing else changes, because week 1's
`providers.py` hides the differences. *A great engine in a bad car goes nowhere; a good car can take another engine.*

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 6 (the same code as the cells above), so you can continue here.
# from checkpoint import *

## 6. The doors: one engine, many ways in (slides 10–12)

Because the engine is headless, the same `Harness` can sit behind any door:

| Door | Who starts it | What happens to the events | Who approves |
|---|---|---|---|
| 1. Interactive | a person in a chat | streamed to their screen, live | the person, right now |
| 2. Remote | a script, an event, a clock | saved for later | nobody is there: changes are **held** |
| 3. Evals | a test run | scored and recorded | a rule decides (section 7) |

### 6a. Door 1, interactive: you steer while it works (slide 11)

You can type while the agent works: *"only refund the second charge"*. The message must not be lost, and it must not
break the step in progress. So:

- a **steering queue**: new messages wait in line;
- a **priority gate**: lets them in at a safe moment (between two steps, never in the middle of a tool call),
  urgent ones (like *stop*) first.

The harness already calls `steering.let_in()` at the top of each turn. Here is the queue:

In [ ]:
URGENT = ("stop", "cancel", "wait")


@dataclass(order=True)
class Message:
    priority: int                                 # 0 = urgent, 1 = normal: lower goes first
    n: int                                        # arrival order, among the same priority
    text: str = field(compare=False)

    @property
    def urgent(self) -> bool:
        return self.priority == 0

    @property
    def is_stop(self) -> bool:
        return self.urgent and self.text.strip().lower().rstrip("!. ") in ("stop", "cancel")


class SteeringQueue:
    """The steering queue + the priority gate. Safe to call put() from another thread (a chat UI)."""

    def __init__(self):
        self.q, self.count = queue.PriorityQueue(), itertools.count()

    def put(self, text: str) -> None:
        urgent = text.strip().lower().rstrip("!. ").split(" ")[0] in URGENT
        self.q.put(Message(0 if urgent else 1, next(self.count), text))

    def waiting(self) -> bool:
        return not self.q.empty()

    def let_in(self) -> list[Message]:
        """Called by the harness between two steps: everything waiting, urgent first."""
        out = []
        while not self.q.empty():
            out.append(self.q.get_nowait())
        return out

🧪 **Alex changes his mind mid-task.** Same Northwind job. While the agent is checking the calendar, Alex types a
correction. (A script plays Alex: it types when it sees the first action, like a person watching the stream.)

In [ ]:
fresh_week()
steer = SteeringQueue()
alex = Harness(ALL_TOOLS, system=SYSTEM, modules=[SKILLS], policy=ALEX_POLICY, approver=always("yes"),
               sandbox=sqlite_sandbox, steering=steer)

typed = []
def alex_types(ev):
    if ev.kind == "action" and not typed:
        typed.append(True)
        steer.put(f"Not {WEEK.label(1)}, please: I'm taking Sam to the doctor that morning. Use {WEEK.label(3)}.")

events = run(alex, TASK, on_event=alex_types)

In [ ]:
turn_table(events)

The correction waited in the queue until the current step finished, then went in as a normal message: the model saw it
on its next turn and booked Thursday. Nothing was cut in half.

**The priority gate.** Two messages are waiting: a normal one, then an urgent *stop*. The gate lets *stop* in first,
and the run ends at the next safe moment.

In [ ]:
fresh_week()
steer = SteeringQueue()
review = Harness(ALL_TOOLS, system=SYSTEM, modules=[SKILLS], policy=ALEX_POLICY, approver=always("no"),
                 steering=steer)

def alex_hurries(ev):
    if ev.kind == "observation" and steer.q.qsize() == 0 and not typed2:
        typed2.append(True)
        steer.put("Also look at next week, please.")
        steer.put("stop")

typed2 = []
events = run(review, "Do my weekly review for this week.", on_event=alex_hurries)

**Your turn (optional, VS Code):** a real chat door. Run the next cell, type a request, and answer the approval
prompts. Type `quit` to leave. (A real chat app would let you type *while* it works: a second thread calls
`steer.put()`.)

In [ ]:
def chat():
    h = Harness(ALL_TOOLS, system=SYSTEM, modules=[SKILLS, memory_module()], policy=ALEX_POLICY,
                approver=ask_human, sandbox=sqlite_sandbox)
    while (text := input("\nYou: ").strip()).lower() not in ("quit", "exit", ""):
        run(h, text)

# chat()      # ← remove the # to try it

### 6b. Door 2, remote: nobody at the keyboard (slide 12)

*Every morning a cron trigger starts one harness per overnight email; each drafts a reply for a human to approve.*
That's the slide's example, and Alex has three emails waiting.

Two things change when nobody is at the keyboard:

1. **Approvals can't wait for an answer.** The approver says **hold**: the change is tried in the sandbox, rolled back,
   and kept in a list with its preview. Alex approves the list in the morning.
2. **Many harnesses can run at once.** They're independent; each runs in its own thread.

In [ ]:
def triage_task(email: dict) -> str:
    return (f"New email from {email['from']}, subject '{email['subject']}':\n\n{email['body']}\n\n"
            "Handle it for Alex: update the calendar if needed and draft a reply if one is needed. "
            "Alex reviews everything in the morning.")


def remote_harness(name: str, **overrides) -> Harness:
    settings = dict(system=SYSTEM, name=name, modules=[SKILLS], policy=ALEX_POLICY, approver=always("hold"),
                    sandbox=sqlite_sandbox)
    return Harness(ALL_TOOLS, **(settings | overrides))


def morning_triage(emails: list[dict], *, quiet: bool = False, **overrides) -> dict[str, tuple[Harness, list]]:
    """The cron job: one headless harness per email, in parallel."""
    harnesses = {e["id"]: remote_harness(e["id"], **overrides) for e in emails}
    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = {e["id"]: pool.submit(run, harnesses[e["id"]], triage_task(e), quiet=quiet, tag=True)
                   for e in emails}
        return {k: (harnesses[k], f.result()) for k, f in futures.items()}


def approve(held: Held, tools: dict = ALL_TOOLS) -> str:
    """Alex said yes in the morning: do it for real now (it can still fail if the calendar changed)."""
    return execute_tool(ToolCall("approved", held.tool, held.args), tools).content

In [ ]:
fresh_week()
t0 = time.time()
overnight = morning_triage(life_tools.inbox())
print(f"\n⏱ {len(overnight)} harnesses in parallel: {time.time() - t0:.0f} s in total")

In [ ]:
print("☕ Alex's morning review\n")
for name, (h, events) in overnight.items():
    print(f"[{name}] {final(events)}")
    for held in h.held:
        print(f"   ⏸ {held.tool}:\n{held.preview}")
print("\nDrafts:", [m["subject"] for m in life_tools.outbox()])

In [ ]:
# Alex approves everything held, in one go:
for h, _ in overnight.values():
    for held in h.held:
        print("✅", approve(held))

The harnesses ran **at the same time**: their lines are interleaved, and the total time is about the time of the
slowest one. Look at the school email: it needs no reply and the form is already on Alex's task list. Did its harness
leave things alone, or add something Alex now has to review? Knowing when *not* to act is part of the job (and a
check in section 7).

**The triggers.** A *cron* line runs the job on a schedule; a *CLI* lets a script or a CI job start it; a *webhook*
starts it when an event arrives. All three call the same engine. A CLI door is a few lines: it imports the
harness we built here (`checkpoint.py` has all the notebook's code).

In [ ]:
%%writefile alex_cli.py
"""Door 2 from a terminal:  python alex_cli.py "What's on Thursday?"   or   python alex_cli.py --triage
Nobody is at the keyboard, so changes are held for Alex. Cron, every weekday at 7:00:
    0 7 * * 1-5  cd /path/to/week-04-agent-harness && ../.venv/bin/python alex_cli.py --triage >> triage.log
"""
import sys

from checkpoint import *          # the harness from the notebook

if sys.argv[1:] == ["--triage"]:
    for name, (h, events) in morning_triage(life_tools.inbox(), quiet=True).items():
        print(f"[{name}] {final(events)}  ({len(h.held)} change(s) held)")
else:
    run(remote_harness("cli"), " ".join(sys.argv[1:]))

In [ ]:
import sys
!{sys.executable} alex_cli.py "What's on my calendar on Thursday? One line per event."

*Optional:* **a webhook door.** A tiny web server: an email arrives as an HTTP POST, a harness handles it, the reply
says what's waiting for Alex. (Real email services can call a URL like this for every new message.)

In [ ]:
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import Request, urlopen


class EmailWebhook(BaseHTTPRequestHandler):
    def do_POST(self):
        email = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        h = remote_harness("webhook")
        events = run(h, triage_task(email), quiet=True)
        body = json.dumps({"answer": final(events), "held": [x.tool for x in h.held]}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass


server = ThreadingHTTPServer(("127.0.0.1", 0), EmailWebhook)          # port 0: any free port
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_address[1]}/email"

fresh_week()
reply = urlopen(Request(url, data=json.dumps(BRIGHTLINE).encode(), headers={"Content-Type": "application/json"}),
                timeout=300)
print(json.loads(reply.read()))
server.shutdown()

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 7 (the same code as the cells above), so you can continue here.
# from checkpoint import *

## 7. Door 3: evals and observability, proof it works (slide 13)

| | Question | What we build |
|---|---|---|
| **Replays** | *What exactly went wrong?* | a **flight recorder**: every event, saved as it happens |
| **Benchmarks** | *How good is it?* | a few tasks with checks, and a score |
| **Regressions** | *Did my change break it?* | rerun the benchmark after every change |

Replays work because **every action goes through the harness**: the loop already produces the log, as events.
The recorder just writes them down, one JSON line each, *as they happen*, so even a crashed run leaves a record.

In [ ]:
RUNS = Path("runs")


def record(events: Iterator[Event], path: Path) -> Iterator[Event]:
    """The flight recorder: pass every event through, and append it to a JSONL file on the way."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for ev in events:
            f.write(json.dumps(asdict(ev), ensure_ascii=False, default=str) + "\n")
            f.flush()
            yield ev


def replay(path: Path) -> list[Event]:
    return [Event(**json.loads(line)) for line in path.read_text(encoding="utf-8").splitlines()]


def recorded_run(harness: Harness, task: str, path: Path, **kw) -> list[Event]:
    events = []
    for ev in record(harness.run(task), path):
        events.append(ev)
        if not kw.get("quiet"):
            show(ev, tag=kw.get("tag", False))
    return events

**A tiny benchmark.** Alex's three overnight emails, each with a check that a rule can decide. The checks read the
**events**, the same thing a replay shows: which actions the agent took, with which arguments, and whether they worked.

In [ ]:
def calls(events: list[Event]) -> list[tuple[str, dict, bool]]:
    """(tool, args, worked) for every action: each action is followed by its observation."""
    out, pending = [], []
    for e in events:
        if e.kind == "action":
            pending.append(e.data)
        elif e.kind == "observation" and pending:
            a = pending.pop(0)
            out.append((a["tool"], a["args"], not e.data["error"]))
    return out


def booked(events) -> list[dict]:
    """Calendar changes the agent made or put on hold (both count: Alex approves holds)."""
    return [a for t, a, ok in calls(events) if ok and t in ("create_event", "move_event")]


def wrote(events, tool_name: str) -> bool:
    return any(t == tool_name and ok for t, a, ok in calls(events))


CHECKS = {
    "northwind": lambda ev: (any(b["date"] in (WEEK.day(1), WEEK.day(3)) and "08:00" <= b["start"]
                                 and b["end"] <= "09:00" for b in booked(ev))
                             and wrote(ev, "draft_email") and not wrote(ev, "send_email")),
    "brightline": lambda ev: (any(b["date"] == WEEK.day(4) and b["start"] == "08:00" for b in booked(ev))
                              and wrote(ev, "draft_email") and not wrote(ev, "send_email")),
    "school": lambda ev: not booked(ev) and not wrote(ev, "draft_email") and not wrote(ev, "send_email"),
}


def benchmark(label: str, repeats: int = 3, **overrides) -> dict[str, list[bool]]:
    """Every email, `repeats` times, each through a fresh remote harness, all in parallel.
    Every run is recorded to runs/<label>-<email>-<n>.jsonl. Models vary: one run proves little."""
    jobs = [(e, n) for e in life_tools.inbox() for n in range(1, repeats + 1)]
    with ThreadPoolExecutor(max_workers=9) as pool:
        futures = [(e["id"], n, pool.submit(recorded_run, remote_harness(f"{e['id']}-{n}", **overrides),
                                            triage_task(e), RUNS / f"{label}-{e['id']}-{n}.jsonl", quiet=True))
                   for e, n in jobs]
        results: dict[str, list[bool]] = {}
        for case, n, f in futures:
            results.setdefault(case, []).append(CHECKS[case](f.result()))
    passed = sum(map(sum, results.values()))
    print(f"{label:<16} " + "  ".join(f"{k} {sum(v)}/{len(v)}" for k, v in results.items())
          + f"   total {passed}/{len(jobs)}")
    return results

**A regression check.** A teammate says: *"The skills module costs tokens on every call. Let's remove it."* Before
merging, rerun the benchmark with and without it:

In [ ]:
fresh_week()
with_skills = benchmark("with skills")
without = benchmark("without skills", modules=[])

Compare the two lines. If a case passes less often without skills, removing them is a **regression**: **replay** a
failed run to see why. If the scores are the same, the skills don't earn their tokens *on these cases*, which is also
worth knowing. Either way it's evidence instead of vibes.

Even with 3 repeats, nine runs per side is a smoke test, not proof: run it twice and the numbers move. Week 5 is
about doing this properly: more cases, more repeats, model-graded checks.

In [ ]:
# replay the first failure (or, if all passed, one run anyway)
failures = [f"{label}-{k}-{n}" for label, res in (("without skills", without), ("with skills", with_skills))
            for k, oks in res.items() for n, ok in enumerate(oks, 1) if not ok]
name = failures[0] if failures else "without skills-brightline-1"
print("Replaying", name)
turn_table(replay(RUNS / f"{name}.jsonl"))

*Crash tests before release, and a flight recorder after.* Tools like Opik or Kitaru do the same thing at scale:
they record the events (traces), replay them, and score them.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 8 (the same code as the cells above), so you can continue here.
# from checkpoint import *

## 8. Same harness shape, different parts plugged in (slide 14)

| Agent | Its tools | Needs approval for | Usually started by |
|---|---|---|---|
| Coding | files, tests, terminal | deleting, pushing code | a developer in a terminal |
| Support | CRM, orders, email | refunds, account changes | a new ticket (webhook) |
| Research | web search, PDFs, notes | paid databases | a question in chat |
| **Assistant (Alex)** | calendar, tasks, email, notes | changing the calendar, sending email | a chat, or cron at 7:00 |

**What stays the same:** the loop, the context window, compaction, permissions, the doors and evals. To prove it, here
is slide 6's support agent: *"I was charged twice"*, with the **same `Harness` class**. Only the modules change:
four fake tools, a refund rule, a support lead who approves.

In [ ]:
CHARGES = {"jordan@example.com": [
    {"charge": "ch_101", "date": "Sep 3", "amount": 49.00, "item": "Pro plan, monthly"},
    {"charge": "ch_102", "date": "Sep 3", "amount": 49.00, "item": "Pro plan, monthly"},
    {"charge": "ch_087", "date": "Aug 3", "amount": 49.00, "item": "Pro plan, monthly"}]}
REFUNDED, SENT = [], []


class EmailIn(Args):
    email: str


class RefundIn(Args):
    charge: str
    reason: str


class ReplyIn(Args):
    email: str
    text: str


def lookup_orders(email: str) -> str:
    return json.dumps(CHARGES.get(email, []))

def refund_policy() -> str:
    return "Duplicate charges are always refunded in full. Refunds over $500 need a manager."

def refund(charge: str, reason: str) -> str:
    REFUNDED.append(charge)
    return f"Refund confirmed for {charge}."

def reply_to_customer(email: str, text: str) -> str:
    SENT.append(text)
    return "Email sent."


SUPPORT_TOOLS = {
    "lookup_orders": tool(lookup_orders, input_model=EmailIn, description="A customer's recent charges."),
    "refund_policy": tool(refund_policy, input_model=Args, description="The refund policy."),
    "refund": tool(refund, input_model=RefundIn, effect=WRITE, description="Refund one charge."),
    "reply_to_customer": tool(reply_to_customer, input_model=ReplyIn, effect=WRITE,
                              description="Email the customer. Ends the ticket."),
}

support = Harness(SUPPORT_TOOLS, name="support",
                  system="You are a support agent for a software company. Check the facts with your tools, "
                         "follow the refund policy, then reply to the customer and stop.",
                  policy=policy_by_effect(overrides={"reply_to_customer": "allow"}),     # refunds need a human
                  approver=always("yes"), sandbox=no_sandbox, extra={"parallel_tool_calls": False})
events = run(support, "New ticket from jordan@example.com: 'I was charged twice this month!'", quiet=True)
turn_table(events)
print("Refunded:", REFUNDED)

Compare with slide 6: look up the orders → check the policy → refund (**paused for a human**) → reply → stop.
The model never touched the payment system. Every action went through the harness, which is where the approval
and the log live. *Add a row for your own capstone agent: what are its tools, what needs approval, what starts it?*

## 9. Case study: a real harness (slide 16)

The coding-agent diagram from the talk (Paul Iusztin, *Decoding AI*), box by box, with what we built today:

| Diagram | Our code |
|---|---|
| **Headless harness** | `Harness`: no screen, `run()` yields `Event`s |
| Context window `[summary, *tail]` + conversation | `harness.conv.input`, `SummaryTail` |
| Compact / should compact? | `SummaryTail.__call__`: threshold → `[summary]` + `[tail]` |
| Agent loop: LLM → Action → Tools → Observation | `run()`: `model` → `action` → `act()` → `observation` |
| Modules: LLM providers | `provider=` / `model=` (week 1's `providers.py`) |
| Modules: memory, skills | `memory_module()`, `skills_module()` |
| Modules: sandbox, permissions | `sqlite_sandbox`, `policy_by_effect` + approvers |
| Modules: LSP server | *coding-specific: for Alex it's the calendar and email tools* |
| 1. Interactive: TUI, steering queue, priority gate, streamed events | `run()` + `show`, `SteeringQueue.let_in()`, `chat()` |
| 2. Remote: CLI, webhook, cron, in parallel, runtime | `alex_cli.py`, `EmailWebhook`, the cron line, `morning_triage` (threads) |
| 3. Evals & observability: benchmarks, regressions, replays | `benchmark`, rerun after a change, `record` / `replay` |

Swap the modules for a CRM and email, and it's a support-agent harness (section 8). Swap them for files, tests and a
terminal, and it's a coding agent.

## Takeaways (slide 15)

> **Same model. Better harness. Better agent.**
> The loop does the work. The context is the craft. Evals tell you the truth.

**Office hours (slide 17):**
1. Which part would you build first for your own agent, and why?
2. Where does your agent go wrong today: the loop, the context or the tools?
3. Which action should never run without a human saying yes?
4. If you could replay one failed run, what would you look for?

## Homework

1. **Package it.** Move the notebook's harness into a `harness/` package (no notebook code left behind):
   ```
   harness/
     events.py        Event, Limits, step_cost
     core.py          Harness (run + act), Held
     context.py       desk_tokens, step_starts, SummaryTail
     modules/         permissions.py (policy, approvers), sandbox.py, skills.py, memory.py
     doors/           interactive.py (SteeringQueue, chat), remote.py (morning_triage + the CLI), webhook.py
     observe.py       show, turn_table, record, replay, calls, benchmark
   ```
   Add `tests/` that run **without an API key**: a fake conversation that plays a script of model steps. At least:
   a `deny` rule never runs the tool; a `hold` leaves the database unchanged and lands in `harness.held`; a declined
   change is rolled back; `stop` from the steering queue ends the run before the next model call, and goes before a
   normal message; `Limits(max_turns=2)` stops at turn 2; `step_starts` never cuts between a call and its result;
   `record` → `replay` gives the same events.
2. **A new door.** Add a fourth trigger to the remote door: a "Sunday 19:00" job that runs the `weekly-review` skill
   and saves the answer to `reviews/<date>.md`. Show the crontab line.
3. **A new check.** Add a fourth email to the benchmark (write it yourself) and a check for it. Run the benchmark three
   times: does it pass every time?

**Stretch:** a `max_cost` limit per *day* across all harnesses (a shared budget) · a `deny` list that the model can
see in its instructions, so it doesn't even try · stream events to a file that another terminal tails live (`tail -f`).